In [1]:
import numpy as np
import matplotlib.pyplot as plt
import refractiveindex as ri


In [2]:
def interface_matrix (n1, n2):
    """Calculates the Fresnel matrix for a single boundary between two media."""
    
    # Fresnel coefficients for reflection (r) and transmission (t)
    r = (n1-n2)/(n1+n2)
    t = (2*n1)/(n1+n2)

    # Standard 2x2 TMM interface representation
    return (1/t) * np.array([
        [1, r],
        [r, 1]
    ], dtype=complex)


In [3]:
def propagation_matrix (n, d, wavelength):
    """Calculates the phase shift as light moves through a layer of thickness d."""

    phi = (2*np.pi*n*d)/wavelength

    return np.array([
        [np.exp(-1j*phi), 0],
        [0, np.exp(1j*phi)]
    ], dtype=complex)


In [4]:
def stack_calculation(layers, wavelength_nm):
    """Multiplies all matrices to get the total system response."""

    M_total = np.eye(2, dtype=complex) # Start with an identity matrix
    wavelength_micron = wavelength_nm/1000 # Convert nm to microns to use the 'ri' library
    n_prev = 1 # Ambient medium is usually Air so the value is 1

    for material, d_layer in layers:
        # Get refractive index from the library or use a fixed number
        if hasattr(material, 'get_refractive_index'):
            n_layer = material.get_refractive_index(wavelength_micron)
        else:
            n_layer = material
        # Matrix multiplication to combine both the interface and the propagation
        M_total = M_total @ interface_matrix(n_prev, n_layer)
        M_total = M_total @ propagation_matrix(n_layer, d_layer, wavelength_nm)
        n_prev = n_layer
        
    # Final boundary back to Air or Substrate
    n_exit = 1
    M_total = M_total @ interface_matrix(n_prev, n_exit)

    r_coeff = M_total[1, 0] / M_total[0, 0] # Reflection coefficient from the complex system matrix

    R = np.abs(r_coeff)**2 # Reflectance (R) is the square of the amplitude
    T = 1 - R # 
    

    return R, T


In [5]:
def plot_sensitivity(layers, error_nm=2, iterations=10):
    """Simulates small errors in thickness to check if the design is stable."""
    
    wavelengths = np.linspace(400, 800, 400)
    plt.figure(figsize=(10, 6))

    for i in range(iterations):
        noisy_layers = [] # Create a "noisy" version of the stack with random errors
        for mat, d in layers:
            # Add a random error within the +/- tolerance range
            error = np.random.uniform(-error_nm, error_nm)
            noisy_layers.append((mat, d + error))

        R_list = [stack_calculation(noisy_layers, wl)[0] * 100 for wl in wavelengths] # Calculate the reflectance spectrum for this specific iteration

        plt.plot(wavelengths, R_list, color='red', alpha=0.3) # Plot with transparency to see the spread of results
        
    plt.title(f"Sensitivity Analysis (Manufacturing Error: ±{error_nm}nm)")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Reflectance (%)")
    plt.grid(True, alpha=0.2)
    plt.show()
    

In [ ]:
from interface import launch_interface
launch_interface(plot_sensitivity)